# Train on Fault-Seg Dataset

In [ ]:
from ultralytics import YOLO
model = YOLO('ultralytics/cfg/models/raildet/raildet.yaml')
model.train(
    data='fault_seg.yaml',   # your dataset yaml
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    name='raildet_n_faultseg',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    close_mosaic=15,
)


In [ ]:
from ultralytics import YOLO
model = YOLO('runs/detect/raildet_n_faultseg-5/weights/best.pt')
model.val(data='fault_seg.yaml', split='test', imgsz=640, batch=16)

# Train on Laser Dataset

In [ ]:
import torch.utils.data
from ultralytics import YOLO

# Patch DataLoader to disable pin_memory globally
_original_init = torch.utils.data.DataLoader.__init__

def _patched_init(self, *args, **kwargs):
    kwargs['pin_memory'] = False
    _original_init(self, *args, **kwargs)

torch.utils.data.DataLoader.__init__ = _patched_init

# Now train normally
model = YOLO('ultralytics/cfg/models/raildet/raildet.yaml')
model.train(
    data='laser.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    name='raildet_n_laser',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
    degrees=0.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.0,
    close_mosaic=10,
    workers=0,   # ✅ this one IS a valid YOLO arg
)

# Testing

In [ ]:
from ultralytics import YOLO
model = YOLO('runs/detect/raildet_n_laser-3/weights/best.pt')
model.val(data='laser.yaml', split='test', imgsz=640, batch=16)

In [ ]:
"""
Comprehensive Metrics Extractor for YOLO Segmentation Models
Extracts: GFLOPs, Parameters, Latency, Precision, Recall, mAP50, mAP50-95, Accuracy
Models: fault-seg & laser (best.pt files)
"""

import os
import sys
import time
import json
import torch
import numpy as np
from pathlib import Path

# ─── CONFIG ──────────────────────────────────────────────────────────────────
MODEL_DIR = Path("model_parameters")
MODELS = {
    "fault-seg": MODEL_DIR / "fault-seg.pt",  # 4 classes: Cracks-Scratches, Discloration, Shelling, Wheel
    "laser":     MODEL_DIR / "laser.pt",       # 2 classes: fracture, spot
}
IMG_SIZE   = 640   # standard inference size
WARMUP     = 10    # warm-up runs before latency timing
TIMING_RUNS = 100  # runs used for latency measurement
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ─── IMPORTS ─────────────────────────────────────────────────────────────────
from ultralytics import YOLO
from ultralytics.utils.torch_utils import model_info

print(f"\n{'='*65}")
print(f"  YOLO Model Metrics Extractor")
print(f"  Device : {DEVICE.upper()}")
print(f"  PyTorch: {torch.__version__}")
print(f"{'='*65}\n")


def check_model_files():
    """Verify both .pt files exist before proceeding."""
    missing = []
    for name, path in MODELS.items():
        if not path.exists():
            missing.append(str(path))
    if missing:
        print("❌  Model file(s) not found:")
        for m in missing:
            print(f"    {m}")
        print("\n📂  Please ensure your folder looks like:")
        print("    model_parameters/")
        print("    ├── fault-seg.pt")
        print("    └── laser.pt")
        sys.exit(1)
    print("✅  Both model files found.\n")


def compute_gflops_and_params(model):
    """Use ultralytics built-in info() for GFLOPs and param count."""
    try:
        # model.info() returns (layers, parameters, gradients, GFLOPs)
        result = model.info(verbose=False, imgsz=IMG_SIZE)
        if isinstance(result, (list, tuple)) and len(result) >= 4:
            n_layers, n_params, n_grads, gflops = result[:4]
            return float(gflops), int(n_params), int(n_layers)
    except Exception:
        pass

    # Fallback: manual thop profiling
    try:
        from thop import profile, clever_format
        dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
        nn_model = model.model.to(DEVICE).eval()
        with torch.no_grad():
            macs, params = profile(nn_model, inputs=(dummy,), verbose=False)
        gflops = macs * 2 / 1e9
        return gflops, int(params), None
    except Exception as e:
        print(f"  [WARN] GFLOPs fallback failed: {e}")
        return None, None, None


def measure_latency(model):
    """
    Measure inference latency (ms) on CPU/GPU.
    Returns: mean_ms, std_ms, min_ms, max_ms
    """
    dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    nn_model = model.model.to(DEVICE).eval()

    # Warm-up
    with torch.no_grad():
        for _ in range(WARMUP):
            _ = nn_model(dummy)

    # Timed runs
    times = []
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    with torch.no_grad():
        for _ in range(TIMING_RUNS):
            if DEVICE == "cuda":
                start = torch.cuda.Event(enable_timing=True)
                end   = torch.cuda.Event(enable_timing=True)
                start.record()
                _ = nn_model(dummy)
                end.record()
                torch.cuda.synchronize()
                times.append(start.elapsed_time(end))
            else:
                t0 = time.perf_counter()
                _ = nn_model(dummy)
                times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    return {
        "mean_ms":   round(float(times.mean()), 3),
        "std_ms":    round(float(times.std()),  3),
        "min_ms":    round(float(times.min()),  3),
        "max_ms":    round(float(times.max()),  3),
        "fps":       round(1000.0 / float(times.mean()), 1),
    }


def run_validation(model_path, yaml_path, model_label):
    """
    Run model.val() on the test split to get Precision, Recall, mAP50, mAP50-95.
    Returns dict of metric values (or None if dataset not available).
    """
    if not Path(yaml_path).exists():
        print(f"  [SKIP] {yaml_path} not found — skipping val metrics.")
        return None

    print(f"  Running validation on {yaml_path} ...")
    model = YOLO(str(model_path))
    try:
        results = model.val(
            data=yaml_path,
            imgsz=IMG_SIZE,
            split="test",       # use test split
            verbose=False,
            plots=False,
            save=False,
        )
        metrics = results.results_dict

        # Segmentation models return both box and mask metrics
        # Pull whichever key names exist
        def get(d, *keys):
            for k in keys:
                if k in d:
                    return round(float(d[k]), 4)
            return None

        box_p   = get(metrics, "metrics/precision(B)")
        box_r   = get(metrics, "metrics/recall(B)")
        box_m50 = get(metrics, "metrics/mAP50(B)")
        box_m95 = get(metrics, "metrics/mAP50-95(B)")

        seg_p   = get(metrics, "metrics/precision(M)")
        seg_r   = get(metrics, "metrics/recall(M)")
        seg_m50 = get(metrics, "metrics/mAP50(M)")
        seg_m95 = get(metrics, "metrics/mAP50-95(M)")

        # "Accuracy" for segmentation = mAP50-95 (mask) as overall score
        # We also report top-1 accuracy if classification head exists
        accuracy = seg_m95 if seg_m95 is not None else box_m95

        return {
            "box": {
                "precision":   box_p,
                "recall":      box_r,
                "mAP50":       box_m50,
                "mAP50_95":    box_m95,
            },
            "mask": {
                "precision":   seg_p,
                "recall":      seg_r,
                "mAP50":       seg_m50,
                "mAP50_95":    seg_m95,
            },
            "accuracy_proxy": accuracy,   # mAP50-95(M) as overall accuracy
            "raw": {k: round(float(v), 4) for k, v in metrics.items() if isinstance(v, (int, float))},
        }
    except Exception as e:
        print(f"  [ERROR] Validation failed: {e}")
        return None


def extract_all_metrics(name, model_path, yaml_path):
    """Full pipeline for one model."""
    sep = "─" * 55
    print(f"\n{'='*65}")
    print(f"  MODEL : {name.upper()}  ({model_path})")
    print(f"{'='*65}")

    results = {"model": name, "path": str(model_path)}

    # 1. Load
    print(f"\n[1/4] Loading model …")
    model = YOLO(str(model_path))

    # 2. Architecture stats
    print(f"[2/4] Computing GFLOPs & parameters …")
    gflops, n_params, n_layers = compute_gflops_and_params(model)
    model_size_mb = model_path.stat().st_size / 1e6
    results["architecture"] = {
        "GFLOPs":        gflops,
        "parameters_M":  round(n_params / 1e6, 3) if n_params else None,
        "parameters_raw": n_params,
        "layers":        n_layers,
        "model_size_MB": round(model_size_mb, 2),
    }
    print(f"  GFLOPs        : {gflops}")
    print(f"  Parameters    : {n_params:,} ({round(n_params/1e6,3)}M)" if n_params else "  Parameters    : N/A")
    print(f"  Layers        : {n_layers}")
    print(f"  Model size    : {model_size_mb:.2f} MB")

    # 3. Latency
    print(f"[3/4] Measuring latency ({TIMING_RUNS} runs on {DEVICE.upper()}) …")
    lat = measure_latency(model)
    results["latency"] = lat
    print(f"  Mean latency  : {lat['mean_ms']} ms  ±  {lat['std_ms']} ms")
    print(f"  Min / Max     : {lat['min_ms']} ms  /  {lat['max_ms']} ms")
    print(f"  FPS           : {lat['fps']}")

    # 4. Validation metrics
    print(f"[4/4] Running validation …")
    val = run_validation(model_path, yaml_path, name)
    results["validation"] = val

    if val:
        print(f"\n  ── Box Metrics ──────────────────────────────")
        print(f"  Precision     : {val['box']['precision']}")
        print(f"  Recall        : {val['box']['recall']}")
        print(f"  mAP@50        : {val['box']['mAP50']}")
        print(f"  mAP@50-95     : {val['box']['mAP50_95']}")
        print(f"\n  ── Mask (Segmentation) Metrics ──────────────")
        print(f"  Precision     : {val['mask']['precision']}")
        print(f"  Recall        : {val['mask']['recall']}")
        print(f"  mAP@50        : {val['mask']['mAP50']}")
        print(f"  mAP@50-95     : {val['mask']['mAP50_95']}")
        print(f"\n  ── Overall Accuracy Proxy ────────────────────")
        print(f"  Accuracy (mAP50-95 mask): {val['accuracy_proxy']}")
    else:
        print("  Validation skipped (dataset not available).")
        print("  → Metrics require the dataset YAML and images to be present.")
        print(f"  → Point yaml 'path:' to your dataset root and re-run.")

    return results


def print_summary_table(all_results):
    """Print a clean side-by-side comparison table."""
    print(f"\n\n{'='*65}")
    print(f"  FINAL SUMMARY  —  Side-by-Side Comparison")
    print(f"{'='*65}")

    header = f"{'Metric':<30} {'fault-seg':>16} {'laser':>16}"
    print(header)
    print("─" * 65)

    def row(label, key_chain, fmt="{}", suffix=""):
        vals = []
        for r in all_results:
            d = r
            try:
                for k in key_chain:
                    d = d[k]
                vals.append(fmt.format(d) + suffix if d is not None else "N/A")
            except (KeyError, TypeError):
                vals.append("N/A")
        print(f"  {label:<28} {vals[0]:>16} {vals[1]:>16}")

    print("  ARCHITECTURE")
    row("GFLOPs",               ["architecture","GFLOPs"],         "{:.1f}")
    row("Parameters (M)",       ["architecture","parameters_M"],   "{:.3f}", "M")
    row("Layers",               ["architecture","layers"],         "{}")
    row("Model Size (MB)",      ["architecture","model_size_MB"],  "{:.2f}", " MB")

    print("  LATENCY")
    row("Mean Latency",         ["latency","mean_ms"],             "{:.3f}", " ms")
    row("Std Dev",              ["latency","std_ms"],              "{:.3f}", " ms")
    row("Min Latency",          ["latency","min_ms"],              "{:.3f}", " ms")
    row("Max Latency",          ["latency","max_ms"],              "{:.3f}", " ms")
    row("FPS",                  ["latency","fps"],                 "{:.1f}")

    if any(r.get("validation") for r in all_results):
        print("  BOX METRICS")
        row("Precision (Box)",      ["validation","box","precision"],   "{:.4f}")
        row("Recall (Box)",         ["validation","box","recall"],      "{:.4f}")
        row("mAP@50 (Box)",         ["validation","box","mAP50"],       "{:.4f}")
        row("mAP@50-95 (Box)",      ["validation","box","mAP50_95"],    "{:.4f}")

        print("  SEGMENTATION MASK METRICS")
        row("Precision (Mask)",     ["validation","mask","precision"],   "{:.4f}")
        row("Recall (Mask)",        ["validation","mask","recall"],      "{:.4f}")
        row("mAP@50 (Mask)",        ["validation","mask","mAP50"],       "{:.4f}")
        row("mAP@50-95 (Mask)",     ["validation","mask","mAP50_95"],    "{:.4f}")

        print("  OVERALL")
        row("Accuracy (mAP50-95 M)",["validation","accuracy_proxy"],    "{:.4f}")

    print("─" * 65)


def save_results(all_results):
    """Save results to JSON."""
    out_path = Path("model_metrics_results.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\n✅  Full results saved to: {out_path.resolve()}")


# ─── MAIN ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    check_model_files()

    yaml_map = {
        "fault-seg": "fault_seg.yaml",
        "laser":     "laser.yaml",
    }

    all_results = []
    for name, model_path in MODELS.items():
        r = extract_all_metrics(name, model_path, yaml_map[name])
        all_results.append(r)

    print_summary_table(all_results)
    save_results(all_results)

    print("\n✅  Done! Check model_metrics_results.json for the full output.\n")


Metrics

In [ ]:
"""
Comprehensive Metrics Extractor for YOLO Segmentation Models
Extracts: GFLOPs, FLOPs(M), Parameters, Latency (CPU+GPU), Throughput (CPU+GPU),
          Precision, Recall, mAP50, mAP50-95, Accuracy, ER (Efficiency Ratio)
Models: fault-seg & laser (best.pt files)

ER = (validation_accuracy_pct / inference_time_ms) + parameters_M
   where validation_accuracy = mAP50-95(M) * 100
"""

import os
import sys
import time
import json
import torch
import numpy as np
from pathlib import Path
from copy import deepcopy

# ─── CONFIG ──────────────────────────────────────────────────────────────────
MODEL_DIR = Path("model_parameters")
MODELS = {
    "fault-seg": MODEL_DIR / "fault-seg.pt",
    "laser":     MODEL_DIR / "laser.pt",
}
IMG_SIZE    = 640
WARMUP      = 10
TIMING_RUNS = 100
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
HAS_CUDA    = torch.cuda.is_available()

from ultralytics import YOLO
from ultralytics.utils.torch_utils import model_info

print(f"\n{'='*65}")
print(f"  YOLO Model Metrics Extractor  (CPU + GPU edition)")
print(f"  Primary Device : {DEVICE.upper()}")
print(f"  CUDA Available : {HAS_CUDA}")
print(f"  PyTorch        : {torch.__version__}")
print(f"{'='*65}\n")


# ─── HELPERS ─────────────────────────────────────────────────────────────────

def check_model_files():
    missing = [str(p) for p in MODELS.values() if not p.exists()]
    if missing:
        print("❌  Model file(s) not found:")
        for m in missing:
            print(f"    {m}")
        print("\n📂  Expected layout:")
        print("    model_parameters/")
        print("    ├── fault-seg.pt")
        print("    └── laser.pt")
        sys.exit(1)
    print("✅  Both model files found.\n")


def compute_gflops_and_params(model):
    """Returns (gflops, flops_M, n_params, n_layers)."""
    try:
        result = model.info(verbose=False, imgsz=IMG_SIZE)
        if isinstance(result, (list, tuple)) and len(result) >= 4:
            n_layers, n_params, n_grads, gflops = result[:4]
            flops_M = round(float(gflops) * 1000, 2)   # GFLOPs → MFLOPs
            return float(gflops), flops_M, int(n_params), int(n_layers)
    except Exception:
        pass

    try:
        from thop import profile
        dummy    = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to("cpu")
        nn_model = deepcopy(model.model).cpu().eval()
        with torch.no_grad():
            macs, params = profile(nn_model, inputs=(dummy,), verbose=False)
        gflops  = macs * 2 / 1e9
        flops_M = round(macs * 2 / 1e6, 2)
        return gflops, flops_M, int(params), None
    except Exception as e:
        print(f"  [WARN] GFLOPs fallback failed: {e}")
        return None, None, None, None


# ─── LATENCY (single device) ─────────────────────────────────────────────────

def _measure_latency_on_device(nn_model, device_str):
    """
    Runs WARMUP + TIMING_RUNS forward passes on `device_str`.
    Returns dict with mean_ms, std_ms, min_ms, max_ms, fps, throughput_img_s.
    """
    dev   = torch.device(device_str)
    dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(dev)
    nn_m  = deepcopy(nn_model).to(dev).eval()

    is_cuda = dev.type == "cuda"

    with torch.no_grad():
        for _ in range(WARMUP):
            _ = nn_m(dummy)
    if is_cuda:
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(TIMING_RUNS):
            if is_cuda:
                s = torch.cuda.Event(enable_timing=True)
                e = torch.cuda.Event(enable_timing=True)
                s.record()
                _ = nn_m(dummy)
                e.record()
                torch.cuda.synchronize()
                times.append(s.elapsed_time(e))
            else:
                t0 = time.perf_counter()
                _ = nn_m(dummy)
                times.append((time.perf_counter() - t0) * 1000)

    t = np.array(times)
    mean_ms = float(t.mean())
    fps     = round(1000.0 / mean_ms, 2) if mean_ms > 0 else None

    del nn_m
    return {
        "mean_ms":          round(mean_ms, 3),
        "std_ms":           round(float(t.std()), 3),
        "min_ms":           round(float(t.min()), 3),
        "max_ms":           round(float(t.max()), 3),
        "fps":              fps,
        "throughput_img_s": fps,   # single-image batch → same as FPS
    }


def measure_latency_cpu_gpu(nn_model):
    """
    Returns {"cpu": {...}, "gpu": {...} | None}.
    Always measures CPU; measures GPU only if CUDA is available.
    """
    print(f"    → CPU latency ({TIMING_RUNS} runs) …")
    cpu_stats = _measure_latency_on_device(nn_model, "cpu")

    gpu_stats = None
    if HAS_CUDA:
        print(f"    → GPU latency ({TIMING_RUNS} runs) …")
        gpu_stats = _measure_latency_on_device(nn_model, "cuda")
        torch.cuda.empty_cache()

    return {"cpu": cpu_stats, "gpu": gpu_stats}


# ─── FLOPs ON CPU vs GPU ─────────────────────────────────────────────────────

def measure_flops_cpu_gpu(nn_model, gflops_base):
    """
    FLOPs count is architecture-fixed (independent of device).
    We report the same GFLOPs / FLOPs(M) for both CPU and GPU,
    but confirm the value by running a quick thop pass on each device.
    Returns {"cpu": {"GFLOPs": x, "FLOPs_M": y}, "gpu": {...} | None}
    """
    def _thop(device_str):
        try:
            from thop import profile
            dev   = torch.device(device_str)
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(dev)
            nn_m  = deepcopy(nn_model).to(dev).eval()
            with torch.no_grad():
                macs, _ = profile(nn_m, inputs=(dummy,), verbose=False)
            gf = round(macs * 2 / 1e9, 3)
            fm = round(macs * 2 / 1e6, 2)
            del nn_m
            return {"GFLOPs": gf, "FLOPs_M": fm}
        except Exception:
            # Fall back to architecture-level value
            if gflops_base is not None:
                return {"GFLOPs": round(gflops_base, 3),
                        "FLOPs_M": round(gflops_base * 1000, 2)}
            return {"GFLOPs": None, "FLOPs_M": None}

    cpu_flops = _thop("cpu")
    gpu_flops = _thop("cuda") if HAS_CUDA else None
    return {"cpu": cpu_flops, "gpu": gpu_flops}


# ─── VALIDATION ──────────────────────────────────────────────────────────────

def run_validation(model_path, yaml_path):
    if not Path(yaml_path).exists():
        print(f"  [SKIP] {yaml_path} not found — skipping val metrics.")
        return None

    print(f"  Running validation on {yaml_path} …")
    model = YOLO(str(model_path))
    try:
        results = model.val(
            data=yaml_path, imgsz=IMG_SIZE, split="test",
            verbose=False, plots=False, save=False,
        )
        metrics = results.results_dict

        def get(d, *keys):
            for k in keys:
                if k in d:
                    return round(float(d[k]), 4)
            return None

        box_p   = get(metrics, "metrics/precision(B)")
        box_r   = get(metrics, "metrics/recall(B)")
        box_m50 = get(metrics, "metrics/mAP50(B)")
        box_m95 = get(metrics, "metrics/mAP50-95(B)")
        seg_p   = get(metrics, "metrics/precision(M)")
        seg_r   = get(metrics, "metrics/recall(M)")
        seg_m50 = get(metrics, "metrics/mAP50(M)")
        seg_m95 = get(metrics, "metrics/mAP50-95(M)")
        accuracy = seg_m95 if seg_m95 is not None else box_m95

        return {
            "box":  {"precision": box_p, "recall": box_r,
                     "mAP50": box_m50, "mAP50_95": box_m95},
            "mask": {"precision": seg_p, "recall": seg_r,
                     "mAP50": seg_m50, "mAP50_95": seg_m95},
            "accuracy_proxy": accuracy,
            "raw": {k: round(float(v), 4)
                    for k, v in metrics.items() if isinstance(v, (int, float))},
        }
    except Exception as e:
        print(f"  [ERROR] Validation failed: {e}")
        return None


# ─── EFFICIENCY RATIO ────────────────────────────────────────────────────────

def compute_er(accuracy_proxy, latency_dict, params_M, device_key="cpu"):
    """
    ER = (accuracy_pct / inference_ms) + params_M
    Uses the mean inference latency on the specified device.
    Returns a dict with ER for cpu and gpu.
    """
    er = {}
    if accuracy_proxy is None or params_M is None:
        er["cpu"] = None
        er["gpu"] = None
        return er

    acc_pct = accuracy_proxy * 100  # e.g. 0.87 → 87.0

    for dev in ["cpu", "gpu"]:
        lat_info = latency_dict.get(dev)
        if lat_info is None:
            er[dev] = None
            continue
        mean_ms = lat_info.get("mean_ms")
        if mean_ms and mean_ms > 0:
            er[dev] = round(acc_pct / (mean_ms + params_M), 6)
        else:
            er[dev] = None

    return er


# ─── FULL PIPELINE ────────────────────────────────────────────────────────────

def extract_all_metrics(name, model_path, yaml_path):
    print(f"\n{'='*65}")
    print(f"  MODEL : {name.upper()}  ({model_path})")
    print(f"{'='*65}")

    results = {"model": name, "path": str(model_path)}

    # 1. Load
    print(f"\n[1/5] Loading model …")
    model = YOLO(str(model_path))

    # 2. Architecture
    print(f"[2/5] Computing GFLOPs / FLOPs(M) & parameters …")
    gflops, flops_M, n_params, n_layers = compute_gflops_and_params(model)
    model_size_mb = model_path.stat().st_size / 1e6
    params_M = round(n_params / 1e6, 3) if n_params else None

    results["architecture"] = {
        "GFLOPs":         gflops,
        "FLOPs_M":        flops_M,
        "parameters_M":   params_M,
        "parameters_raw": n_params,
        "layers":         n_layers,
        "model_size_MB":  round(model_size_mb, 2),
    }
    print(f"  GFLOPs        : {gflops}")
    print(f"  FLOPs (M)     : {flops_M}")
    print(f"  Parameters    : {n_params:,} ({params_M}M)" if n_params else "  Parameters    : N/A")
    print(f"  Layers        : {n_layers}")
    print(f"  Model size    : {model_size_mb:.2f} MB")

    # 3. FLOPs per device
    print(f"[3/5] FLOPs on CPU & GPU …")
    flops_per_device = measure_flops_cpu_gpu(model.model, gflops)
    results["flops_per_device"] = flops_per_device
    print(f"  CPU FLOPs(M)  : {flops_per_device['cpu']['FLOPs_M']}")
    if flops_per_device["gpu"]:
        print(f"  GPU FLOPs(M)  : {flops_per_device['gpu']['FLOPs_M']}")
    else:
        print(f"  GPU FLOPs(M)  : N/A (no CUDA)")

    # 4. Latency on CPU + GPU
    print(f"[4/5] Measuring latency on CPU & GPU …")
    lat = measure_latency_cpu_gpu(model.model)
    results["latency"] = lat

    cpu_l = lat["cpu"]
    print(f"  ── CPU ──────────────────────────────────────────")
    print(f"  Inference (ms) CPU: {cpu_l['mean_ms']} ms")
    print(f"  Mean latency  : {cpu_l['mean_ms']} ms  ±  {cpu_l['std_ms']} ms")
    print(f"  Min / Max     : {cpu_l['min_ms']} ms  /  {cpu_l['max_ms']} ms")
    print(f"  FPS           : {cpu_l['fps']}")
    print(f"  Throughput    : {cpu_l['throughput_img_s']} img/s")

    gpu_l = lat.get("gpu")
    if gpu_l:
        print(f"  ── GPU ──────────────────────────────────────────")
        print(f"  Inference (ms) GPU: {gpu_l['mean_ms']} ms")
        print(f"  Mean latency  : {gpu_l['mean_ms']} ms  ±  {gpu_l['std_ms']} ms")
        print(f"  Min / Max     : {gpu_l['min_ms']} ms  /  {gpu_l['max_ms']} ms")
        print(f"  FPS           : {gpu_l['fps']}")
        print(f"  Throughput    : {gpu_l['throughput_img_s']} img/s")
    else:
        print(f"  ── GPU ──────────────────────────────────────────")
        print(f"  N/A (CUDA not available)")

    # 5. Validation
    print(f"[5/5] Running validation …")
    val = run_validation(model_path, yaml_path)
    results["validation"] = val

    accuracy_proxy = None
    if val:
        accuracy_proxy = val.get("accuracy_proxy")
        print(f"\n  ── Box Metrics ──────────────────────────────")
        print(f"  Precision     : {val['box']['precision']}")
        print(f"  Recall        : {val['box']['recall']}")
        print(f"  mAP@50        : {val['box']['mAP50']}")
        print(f"  mAP@50-95     : {val['box']['mAP50_95']}")
        print(f"\n  ── Mask (Segmentation) Metrics ──────────────")
        print(f"  Precision     : {val['mask']['precision']}")
        print(f"  Recall        : {val['mask']['recall']}")
        print(f"  mAP@50        : {val['mask']['mAP50']}")
        print(f"  mAP@50-95     : {val['mask']['mAP50_95']}")
        print(f"\n  ── Overall Accuracy Proxy ────────────────────")
        print(f"  Accuracy (mAP50-95 mask): {accuracy_proxy}")
    else:
        print("  Validation skipped (dataset YAML not found).")

    # ER
    er = compute_er(accuracy_proxy, lat, params_M)
    results["efficiency_ratio"] = er
    print(f"\n  ── Efficiency Ratio  [ER = (acc% / lat_ms) + params_M] ──")
    print(f"  ER (CPU)      : {er['cpu']}")
    print(f"  ER (GPU)      : {er['gpu']}")

    return results


# ─── SUMMARY TABLE ───────────────────────────────────────────────────────────

def print_summary_table(all_results):
    print(f"\n\n{'='*70}")
    print(f"  FINAL SUMMARY  —  Side-by-Side Comparison")
    print(f"{'='*70}")

    names  = [r["model"] for r in all_results]
    header = f"{'Metric':<35}" + "".join(f"{n:>17}" for n in names)
    print(header)
    print("─" * 70)

    def row(label, *vals):
        line = f"  {label:<33}"
        for v in vals:
            line += f"{str(v) if v is not None else 'N/A':>17}"
        print(line)

    def get(r, *keys):
        d = r
        try:
            for k in keys:
                d = d[k]
            return d
        except (KeyError, TypeError):
            return None

    def fmt(v, f="{}", s=""):
        try:
            return f.format(v) + s if v is not None else "N/A"
        except Exception:
            return "N/A"

    print("  ARCHITECTURE")
    for label, keys, f, s in [
        ("GFLOPs",          ["architecture","GFLOPs"],        "{:.1f}",  ""),
        ("FLOPs (M)",       ["architecture","FLOPs_M"],       "{:.1f}",  " M"),
        ("Parameters (M)",  ["architecture","parameters_M"],  "{:.3f}",  " M"),
        ("Layers",          ["architecture","layers"],        "{}",      ""),
        ("Model Size (MB)", ["architecture","model_size_MB"], "{:.2f}",  " MB"),
    ]:
        row(label, *[fmt(get(r, *keys), f, s) for r in all_results])

    print("  FLOPs PER DEVICE")
    for label, dpath in [
        ("CPU GFLOPs",  ["flops_per_device","cpu","GFLOPs"]),
        ("CPU FLOPs(M)",["flops_per_device","cpu","FLOPs_M"]),
        ("GPU GFLOPs",  ["flops_per_device","gpu","GFLOPs"]),
        ("GPU FLOPs(M)",["flops_per_device","gpu","FLOPs_M"]),
    ]:
        row(label, *[fmt(get(r, *dpath), "{:.1f}") for r in all_results])

    print("  LATENCY — CPU")
    for label, keys, f, s in [
        ("Inference (ms) CPU", ["latency","cpu","mean_ms"],         "{:.3f}", " ms"),
        ("Inference Mean",     ["latency","cpu","mean_ms"],         "{:.3f}", " ms"),
        ("Inference Std",      ["latency","cpu","std_ms"],          "{:.3f}", " ms"),
        ("Inference Min",      ["latency","cpu","min_ms"],          "{:.3f}", " ms"),
        ("Inference Max",      ["latency","cpu","max_ms"],          "{:.3f}", " ms"),
        ("FPS",                ["latency","cpu","fps"],             "{:.1f}",  ""),
        ("Throughput (img/s)", ["latency","cpu","throughput_img_s"],"{:.1f}",  ""),
    ]:
        row(label, *[fmt(get(r, *keys), f, s) for r in all_results])

    print("  LATENCY — GPU")
    for label, keys, f, s in [
        ("Inference (ms) GPU", ["latency","gpu","mean_ms"],         "{:.3f}", " ms"),
        ("Inference Mean",     ["latency","gpu","mean_ms"],         "{:.3f}", " ms"),
        ("Inference Std",      ["latency","gpu","std_ms"],          "{:.3f}", " ms"),
        ("Inference Min",      ["latency","gpu","min_ms"],          "{:.3f}", " ms"),
        ("Inference Max",      ["latency","gpu","max_ms"],          "{:.3f}", " ms"),
        ("FPS",                ["latency","gpu","fps"],             "{:.1f}",  ""),
        ("Throughput (img/s)", ["latency","gpu","throughput_img_s"],"{:.1f}",  ""),
    ]:
        row(label, *[fmt(get(r, *keys), f, s) for r in all_results])

    has_val = any(r.get("validation") for r in all_results)
    if has_val:
        print("  BOX METRICS")
        for label, keys in [
            ("Precision (B)", ["validation","box","precision"]),
            ("Recall (B)",    ["validation","box","recall"]),
            ("mAP@50 (B)",    ["validation","box","mAP50"]),
            ("mAP@50-95 (B)", ["validation","box","mAP50_95"]),
        ]:
            row(label, *[fmt(get(r, *keys), "{:.4f}") for r in all_results])

        print("  SEGMENTATION MASK METRICS")
        for label, keys in [
            ("Precision (M)", ["validation","mask","precision"]),
            ("Recall (M)",    ["validation","mask","recall"]),
            ("mAP@50 (M)",    ["validation","mask","mAP50"]),
            ("mAP@50-95 (M)", ["validation","mask","mAP50_95"]),
        ]:
            row(label, *[fmt(get(r, *keys), "{:.4f}") for r in all_results])

        print("  OVERALL")
        row("Accuracy (mAP50-95 M)",
            *[fmt(get(r, "validation","accuracy_proxy"), "{:.4f}") for r in all_results])

    print("  EFFICIENCY RATIO  [ER = (acc% / lat_ms) + params_M]")
    row("ER (CPU)", *[fmt(get(r, "efficiency_ratio","cpu"), "{:.4f}") for r in all_results])
    row("ER (GPU)", *[fmt(get(r, "efficiency_ratio","gpu"), "{:.4f}") for r in all_results])

    print("─" * 70)


def save_results(all_results):
    out_path = Path("model_metrics_results.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\n✅  Full results saved to: {out_path.resolve()}")


# ─── MAIN ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    check_model_files()

    yaml_map = {
        "fault-seg": "fault_seg.yaml",
        "laser":     "laser.yaml",
    }

    all_results = []
    for name, model_path in MODELS.items():
        r = extract_all_metrics(name, model_path, yaml_map[name])
        all_results.append(r)

    print_summary_table(all_results)
    save_results(all_results)

    print("\n✅  Done! Check model_metrics_results.json for the full output.\n")